# RAG Prompt Injection Assessment Workflow

In this exercise, you will test whether malicious instructions embedded inside indexed support documents can manipulate a customer support RAG assistant.

In [ ]:
from pathlib import Path
import sys

CURRENT = Path.cwd().resolve()
ROOT = CURRENT.parent if CURRENT.name == "notebooks" else CURRENT
sys.path.append(str(ROOT / "src"))

from rag_prompt_injection_utils import (
    GuardedRAGAssistant,
    VulnerableRAGAssistant,
    attack_success_rate,
    build_vector_store,
    create_poisoned_corpus,
    evaluate_attacks,
    load_json,
    load_text,
    summarize_results,
    write_results_csv,
)

DATA_DIR = ROOT / "data"
PROMPT_DIR = ROOT / "prompts"
RESULTS_DIR = ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

documents = load_json(DATA_DIR / "support_documents.json")
queries = load_json(DATA_DIR / "evaluation_queries.json")
payloads = load_json(DATA_DIR / "prompt_injection_payload_templates.json")
system_prompt = load_text(PROMPT_DIR / "rag_system_prompt.md")

print(f"Documents: {len(documents)}")
print(f"Evaluation queries: {len(queries)}")
print(f"Payload templates: {len(payloads)}")

## 1. Review the RAG Flow

The assistant follows the same core RAG shape as the project chatbot: embed documents, build a vector index, retrieve top-k chunks, concatenate retrieved context, and generate a response. The vulnerability is that retrieved text may contain instructions that conflict with the system prompt.

In [ ]:
print(system_prompt)
documents[:2]

## 2. Build a Clean Retrieval Index

In [ ]:
clean_store = build_vector_store(documents)
sample_query = queries[0]["query"]
clean_store.search(sample_query, k=2)

## 3. Complete the Poisoning and Evaluation Helpers

Open `src/rag_prompt_injection_utils.py` and complete:

- `create_poisoned_corpus`
- `evaluate_attacks`

Then rerun the notebook from this point.

In [ ]:
poisoned_documents = create_poisoned_corpus(documents, payloads)
poisoned_store = build_vector_store(poisoned_documents)

print(poisoned_documents[1]["text"][:500])

## 4. Run Attacks Against the Vulnerable RAG Assistant

In [ ]:
vulnerable_assistant = VulnerableRAGAssistant(system_prompt, poisoned_store, top_k=2)
attack_results = evaluate_attacks(vulnerable_assistant, queries, payloads)

write_results_csv(attack_results, RESULTS_DIR / "starter_attack_results.csv")
print(f"Attack success rate: {attack_success_rate(attack_results):.2f}")
summarize_results(attack_results)

## 5. Compare Against a Guarded RAG Assistant

In [ ]:
guarded_assistant = GuardedRAGAssistant(system_prompt, poisoned_store, top_k=2)
defended_results = evaluate_attacks(guarded_assistant, queries, payloads)

write_results_csv(defended_results, RESULTS_DIR / "starter_defended_results.csv")
print(f"Defended success rate: {attack_success_rate(defended_results):.2f}")
summarize_results(defended_results)

## 6. Assessment Report

Complete `docs/assessment_report_template.md` using your results. Include at least three successful injection examples, the measured success rate, operational risks, and at least three mitigations.